Konteks / Skenario
Platform "Kopi Nusantara" (skenario yang sama dari Tugas Mandiri Pertemuan 7) ingin mulai menyimpan katalog menu mereka di MongoDB — karena setiap cabang punya menu musiman/spesial dengan atribut yang berbeda-beda (tidak semua menu punya atribut yang sama, misalnya hanya menu kopi yang punya atribut "tingkat_kepahitan", sementara menu makanan punya atribut "pedas_level"). Kalian ditugaskan membangun dan menganalisis katalog ini.



A. Membangun Katalog (Create)

In [1]:
from pymongo import MongoClient

# 1. Koneksi ke MongoDB
client = MongoClient("mongodb://localhost:27017/")

# 2. Membuat database 'kopi_nusantara' dan collection 'menu'
db = client["kopi_nusantara"]
collection = db["menu"]

# Hapus data sebelumnya jika ingin dijalankan ulang dari awal secara bersih
collection.drop()

# 3. Menyisipkan dokumen menu (disedirhanakan menjadi 4 dokumen dengan field bervariasi)
data_menu = [
    {
        "nama_menu": "Espresso",
        "harga": 18000,
        "kategori": "Kopi",
        "tingkat_kepahitan": 5,
        "stok": 25,
        "rekomendasi_barista": True,
    },
    {
        "nama_menu": "Cappuccino",
        "harga": 28000,
        "kategori": "Kopi",
        "tingkat_kepahitan": 3,
        "stok": 20,
        "rekomendasi_barista": True,
    },
    {
        "nama_menu": "Croissant",
        "harga": 20000,
        "kategori": "Makanan",
        "pedas_level": 0,
        "stok": 10,
        "rekomendasi_barista": True,
    },
    {
        "nama_menu": "Nasi Goreng Spesial",
        "harga": 35000,
        "kategori": "Makanan",
        "pedas_level": 2,
        "stok": 15,
        "rekomendasi_barista": True,
    },
]

# Masukkan data ke collection
result = collection.insert_many(data_menu)
print(f"Berhasil memasukkan {len(result.inserted_ids)} dokumen ke collection 'menu'.")

Berhasil memasukkan 4 dokumen ke collection 'menu'.


B. Kueri Bertingkat (Read) 

In [2]:
# 1. Seluruh menu kategori "Kopi" dengan tingkat_kepahitan di atas 3
print("--- 1. Kopi dengan tingkat kepahitan > 3 ---")
query_1 = collection.find(
    {"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}}
)
for doc in query_1:
    print(
        f"- {doc['nama_menu']} (Kepahitan: {doc['tingkat_kepahitan']}, Harga: {doc['harga']})"
    )

# 2. Seluruh menu dengan harga di antara 15000 dan 30000
print("\n--- 2. Menu dengan harga antara 15000 dan 30000 ---")
query_2 = collection.find({"harga": {"$gte": 15000, "$lte": 30000}})
for doc in query_2:
    print(f"- {doc['nama_menu']} (Harga: {doc['harga']}, Kategori: {doc['kategori']})")

# 3. Menu yang tidak memiliki field pedas_level sama sekali
print("\n--- 3. Menu yang TIDAK memiliki field pedas_level ---")
query_3 = collection.find({"pedas_level": {"$exists": False}})
for doc in query_3:
    print(f"- {doc['nama_menu']} (Kategori: {doc['kategori']})")

--- 1. Kopi dengan tingkat kepahitan > 3 ---
- Espresso (Kepahitan: 5, Harga: 18000)

--- 2. Menu dengan harga antara 15000 dan 30000 ---
- Espresso (Harga: 18000, Kategori: Kopi)
- Cappuccino (Harga: 28000, Kategori: Kopi)
- Croissant (Harga: 20000, Kategori: Makanan)

--- 3. Menu yang TIDAK memiliki field pedas_level ---
- Espresso (Kategori: Kopi)
- Cappuccino (Kategori: Kopi)


C. Pembaruan Data (Update)

In [3]:
# 1. Tambahkan field 'status' bernilai "Tersedia" ke seluruh dokumen
update_all = collection.update_many({}, {"$set": {"status": "Tersedia"}})
print(f"Jumlah dokumen yang diperbarui statusnya menjadi Tersedia: {update_all.modified_count}")

# 2. Ubah status menjadi "Habis" khusus untuk 2 menu pilihan menggunakan update_one()
collection.update_one({"nama_menu": "Espresso"}, {"$set": {"status": "Habis"}})
collection.update_one({"nama_menu": "Croissant"}, {"$set": {"status": "Habis"}})

print("\nVerifikasi pembaruan status:")
for doc in collection.find({}, {"nama_menu": 1, "status": 1, "_id": 0}):
    print(doc)

Jumlah dokumen yang diperbarui statusnya menjadi Tersedia: 4

Verifikasi pembaruan status:
{'nama_menu': 'Espresso', 'status': 'Habis'}
{'nama_menu': 'Cappuccino', 'status': 'Tersedia'}
{'nama_menu': 'Croissant', 'status': 'Habis'}
{'nama_menu': 'Nasi Goreng Spesial', 'status': 'Tersedia'}


D. Integrasi ke Spark

In [4]:
from pyspark.sql import SparkSession

# Hentikan session lama jika ada
if 'spark' in locals():
    spark.stop()

# Membuat SparkSession dengan konfigurasi input.uri yang benar untuk connector v3.x
spark = SparkSession.builder \
    .appName("TugasMongoDBSpark") \
    .config("spark.mongodb.input.uri", "mongodb://localhost:27017/kopi_nusantara.menu") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1") \
    .getOrCreate()

# 1. Ambil seluruh data dari collection menu menjadi Spark DataFrame
df = spark.read.format("mongo").load()

# Tampilkan beberapa baris data awal
df.show(5, truncate=False)

# 2. Tampilkan printSchema() hasil konversi
print("--- 1. printSchema() ---")
df.printSchema()

# 3. Rata-rata harga per kategori menggunakan groupBy()
print("--- 2. Rata-rata Harga per Kategori ---")
from pyspark.sql.functions import avg

rata_rata_harga = df.groupBy("kategori").agg(
    avg("harga").alias("rata_rata_harga")
)
rata_rata_harga.show()

26/11/04 17:20:09 WARN Utils: Your hostname, adnan resolves to a loopback address: 127.0.1.1; using 192.168.1.6 instead (on interface wlp2s0)
26/11/04 17:20:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/adnan/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/adnan/.ivy2/cache
The jars for the packages stored in: /home/adnan/.ivy2/jars
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-fb601a05-ad81-4ba4-9951-aa4838bdb67f;1.0
	confs: [default]
	found org.mongodb.spark#mongo-spark-connector_2.12;3.0.1 in central
	found org.mongodb#mongodb-driver-sync;4.0.5 in central
	found org.mongodb#bson;4.0.5 in central
	found org.mongodb#mongodb-driver-core;4.0.5 in central
:: resolution report :: resolve 228ms :: artifacts dl 6ms
	:: modules in use:
	org.mongodb#bson;4.0.5 from central in [default]
	org.mongodb#mongodb-driver-core;4.0.5 from central in [default]
	org.mongodb#mongodb-driver-sync;4.0.5 from central in [default]
	org.mongodb.spark#mongo-spark-connector_2.12;3.0.1 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   

+--------------------------+-----+--------+-------------------+-----------+-------------------+--------+----+-----------------+
|_id                       |harga|kategori|nama_menu          |pedas_level|rekomendasi_barista|status  |stok|tingkat_kepahitan|
+--------------------------+-----+--------+-------------------+-----------+-------------------+--------+----+-----------------+
|{6aeb06d80df568092ca8fa8d}|18000|Kopi    |Espresso           |NULL       |true               |Habis   |25  |5                |
|{6aeb06d80df568092ca8fa8e}|28000|Kopi    |Cappuccino         |NULL       |true               |Tersedia|20  |3                |
|{6aeb06d80df568092ca8fa8f}|20000|Makanan |Croissant          |0          |true               |Habis   |10  |NULL             |
|{6aeb06d80df568092ca8fa90}|35000|Makanan |Nasi Goreng Spesial|2          |true               |Tersedia|15  |NULL             |
+--------------------------+-----+--------+-------------------+-----------+-------------------+--------+

E. Kesimpulan 

Mengonversi data dari database NoSQL seperti MongoDB ke dalam Spark DataFrame memiliki beberapa tantangan utama, terutama karena sifat skema dokumen MongoDB yang dinamis dan tidak seragam (schema-less). Berbeda dengan tabel relasional atau Spark DataFrame yang menuntut skema kolom yang kaku dan seragam di setiap barisnya, dokumen di MongoDB dapat memiliki field opsional yang hanya muncul pada dokumen tertentu saja. 

Sebagai contoh pada katalog kopi ini, field seperti `tingkat_kepahitan` hanya dimiliki oleh dokumen dengan kategori "Kopi", sedangkan field `pedas_level` hanya dimiliki oleh dokumen kategori "Makanan". Ketika data yang tidak seragam ini dibaca ke dalam Spark, Spark secara otomatis akan melakukan inferensi skema dengan menggabungkan seluruh field yang ada dan mengisi nilai yang kosong pada dokumen lain dengan nilai `null`. 

Cara saya mengatasi tantangan ini adalah dengan memastikan bahwa proses inferensi skema Spark mengenali tipe data secara konsisten melalui penanganan nilai `null` yang tepat. Jika diperlukan, saya dapat memanfaatkan fungsi bawaan PySpark seperti `.fillna()` untuk mengisi nilai kosong dengan nilai default atau menggunakan fungsi penanganan data bernilai `null` agar operasi agregasi seperti `groupBy` atau analisis lanjutan tidak mengalami distorsi akibat perbedaan struktur data antar dokumen.